# Multi-LLM Debate — Predict & Evaluate

Runs the five-step debate pipeline for Big Five personality prediction.

**Pipeline per record:**
1. Profile the test essay (label-blind) → better query embedding.
2. Retrieve k×2 similar training examples; split randomly: 3 → Model A, 3 → Model B.
3. **Model A** predicts all 5 traits simultaneously with its context. (~80 tokens out)
4. **Model B** predicts all 5 traits independently with its context. (~80 tokens out)
5. Compare predictions → find agreements and disagreements.
6. For every disagreed trait, ask each model: *assumption / evidence / falsifier*. (~600 tokens out)
7. **Verifier** reads both predictions + the debate → `<reasoning>` + `<predictions>`. (~400 tokens out)

**Requires:** A FAISS index — run any `build_*.ipynb` notebook first.

In [6]:
from pathlib import Path
import sys, os, json, random

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ptd_model.predict import (
    predict_debate,
    _retrieve_for_debate,
    _build_all_traits_context,
    _call_profiler_for_text,
    _profile_to_full_text,
)
from ptd_model.evaluate import evaluate

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [7]:
# --- Paths -------------------------------------------------------------------
test_csv      = str(project_root / "data/split/essays/test50.csv")
vector_db_dir = None   # None = auto-detect best available index

res_dir = str(project_root / "result")
log_dir = str(project_root / "log")

# --- Models ------------------------------------------------------------------
model_a_name        = "gpt-4o-mini"   # first predictor
model_b_name        = "gpt-4o-mini"   # second predictor (swap for a different model in real runs)
verifier_model_name = "gpt-4o-mini"        # adjudicator — use the strongest model available

profiler_model = "gpt-4o-mini"        # profiles the test essay for dual embedding

# --- Retrieval ---------------------------------------------------------------
top_k       = 3     # examples per model; retrieves top_k*2 = 6 total, splits 3/3
temperature = 0.0

# --- Misc --------------------------------------------------------------------
TRAIT_TAGS = ("Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")

test_df = pd.read_csv(test_csv)
print(f"Test set : {len(test_df)} rows")
print(f"Model A  : {model_a_name}")
print(f"Model B  : {model_b_name}")
print(f"Verifier : {verifier_model_name}")
print(f"top_k    : {top_k} per model  ({top_k * 2} retrieved total)")

Test set : 50 rows
Model A  : gpt-4o-mini
Model B  : gpt-4o-mini
Verifier : gpt-4o-mini
top_k    : 3 per model  (6 retrieved total)


## Step 1 — Profile the test set (label-blind)

Profiles are generated **without labels** (`use_labels=False`) to prevent leakage.
They are only used to build a richer query embedding — they are never shown to the predictor.

In [8]:
from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles

test_profile_db = str(project_root / "data/profile_db/essays_test")
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store      = ProfileStore(str(test_store_path))
test_store.load()

needed = len(test_df) - sum(
    1 for i in range(len(test_df))
    if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Profiles in store: {len(test_store)}  |  missing: {needed}")

if needed > 0:
    test_store = build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = profiler_model,
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,
    )

print("Test profiles ready.")

Profiles in store: 247  |  missing: 0
Test profiles ready.


## Step 2 — Sanity check: preview one debate context

Retrieves k×2 examples for the first test essay, applies the random 3/3 split,
and prints what each model will see. Confirms the index loads and the split works.

In [9]:
import rag.embedder as _embedder
from rag.retriever import FeatureRAGRetriever

_embedder.FINETUNED_MODEL_DIR = str(project_root / "models" / "sbert_essays_finetuned")
_check_retriever = FeatureRAGRetriever(db_dir=vector_db_dir)

sample_text = test_df.iloc[0]["text"]

# Profile the sample essay for dual embedding
_profile_text = None
try:
    _p = _call_profiler_for_text(sample_text, model_name=profiler_model)
    if _p.get("valid"):
        _profile_text = _profile_to_full_text(_p)
        print("Profile: valid")
    else:
        print("Profile: invalid — raw-text embedding fallback")
except Exception as exc:
    print(f"Profiler error: {exc} — raw-text embedding fallback")

# Retrieve and split
_pool = _retrieve_for_debate(_check_retriever, sample_text, _profile_text, top_k_total=top_k * 2)
random.shuffle(_pool)
_ctx_a = _build_all_traits_context(_pool[:top_k], top_k)
_ctx_b = _build_all_traits_context(_pool[top_k : top_k * 2] or _pool[:top_k], top_k)

print(f"\nRetrieved {len(_pool)} examples → split {top_k} / {top_k}")
print("\n" + "=" * 60)
print("CONTEXT FOR MODEL A")
print("=" * 60)
print(_ctx_a)
print("\n" + "=" * 60)
print("CONTEXT FOR MODEL B")
print("=" * 60)
print(_ctx_b)

[retriever] mode='dual'  dir='F:\\std\\GR\\code\\model_x_ocean\\data\\vector_db\\essays_dual'  hybrid=True
Profile: valid
[retriever] Dual index loaded (1974 vectors).

Retrieved 6 examples → split 3 / 3

CONTEXT FOR MODEL A
[Similar Profile 1] (labels: Openness=high, Conscientiousness=low, Extraversion=low, Agreeableness=low, Neuroticism=high)
N1|high|expresses multiple worries about school, relationships, and personal health.
N2|low|shows irritation towards distractions but no significant anger towards others.
N3|mod|mentions feeling like a "complete piece of crap" but does not express deep sadness.
N4|mod|reflects on feelings about relationships but does not show strong embarrassment.
N5|mod|acknowledges partying too much and needing to establish a routine.
N6|high|feels overwhelmed by school stress and personal issues, indicating a sense of helplessness.
E1|low|lacks affectionate engagement; mentions relationships without emotional depth.
E2|mod|looks forward to fraternity events b

## Step 3 — Run predict_debate

Runs the full debate pipeline over the entire test set.
Saves `predictions.csv` and `debate_log.jsonl` to the output directory.

> **Cost note:** each record triggers up to 5 LLM calls (2 predict + 2 debate + 1 verify).
> At 247 test records that is up to ~1 235 API calls. Debate calls are skipped when all 5 traits agree.

In [10]:
run_id, run_time, prediction_csv = predict_debate(
    text_df             = test_df,
    model_a_name        = model_a_name,
    model_b_name        = model_b_name,
    verifier_model_name = verifier_model_name,
    log_dir             = log_dir,
    res_dir             = res_dir,
    temperature         = temperature,
    top_k               = top_k,
    vector_db_dir       = vector_db_dir,
    profiler_model      = profiler_model,
)

debate_log_path = str(Path(prediction_csv).parent / "debate_log.jsonl")
print(f"\nFinished in {run_time:.1f}s")
print(f"Predictions : {prediction_csv}")
print(f"Debate log  : {debate_log_path}")

[retriever] mode='dual'  dir='F:\\std\\GR\\code\\model_x_ocean\\data\\vector_db\\essays_dual'  hybrid=True
[debate] RAG retriever ready (k_per_model=3).
[debate] 50 records | A=gpt-4o-mini | B=gpt-4o-mini | V=gpt-4o-mini
[retriever] Dual index loaded (1974 vectors).
  [debate] 10/50 done.
  [debate] 20/50 done.
  [debate] 30/50 done.
  [debate] 40/50 done.
  [debate] 50/50 done.
[debate] Finished in 1598.04s -> F:\std\GR\code\model_x_ocean\result\debate\gpt-4o-mini_vs_gpt-4o-mini\20260518-211158\predictions.csv

Finished in 1598.0s
Predictions : F:\std\GR\code\model_x_ocean\result\debate\gpt-4o-mini_vs_gpt-4o-mini\20260518-211158\predictions.csv
Debate log  : F:\std\GR\code\model_x_ocean\result\debate\gpt-4o-mini_vs_gpt-4o-mini\20260518-211158\debate_log.jsonl


## Step 4 — Inspect the debate log

Each line in `debate_log.jsonl` is one record with:
- `preds_a` / `preds_b` — each model's all-traits prediction
- `agree` / `disagree` — which traits matched and which did not
- `final` — the verifier's verdict

The cell below shows aggregate statistics and prints one contested record in full.

In [11]:
with open(debate_log_path, encoding="utf-8") as _f:
    debate_records = [json.loads(line) for line in _f if line.strip()]

total      = len(debate_records)
all_agreed = sum(1 for r in debate_records if not r["disagree"])
print(f"Records           : {total}")
print(f"Full agreement    : {all_agreed}  ({100*all_agreed/total:.1f}%)  — debate skipped")
print(f"Had disagreement  : {total - all_agreed}  ({100*(total-all_agreed)/total:.1f}%)")

# Per-trait disagreement rate
print("\nPer-trait disagreement rate:")
for trait in TRAIT_TAGS:
    n_dis = sum(1 for r in debate_records if trait in r["disagree"])
    print(f"  {trait:<20}: {n_dis}/{total}  ({100*n_dis/total:.1f}%)")

# Print one contested record in full
contested = [r for r in debate_records if r["disagree"]]
if contested:
    ex = contested[0]
    print(f"\n{'='*60}")
    print(f"Example contested record  (idx={ex['record_idx']})")
    print(f"{'='*60}")
    print(f"Agreed on   : {ex['agree']}")
    print(f"Disagreed on: {ex['disagree']}")
    print()
    print("Model A:")
    for t, v in ex["preds_a"].items():
        marker = " ← disputed" if t in ex["disagree"] else ""
        print(f"  {t:<20}: {v}{marker}")
    print()
    print("Model B:")
    for t, v in ex["preds_b"].items():
        marker = " ← disputed" if t in ex["disagree"] else ""
        print(f"  {t:<20}: {v}{marker}")
    print()
    print("Verifier final:")
    for t, v in ex["final"].items():
        print(f"  {t:<20}: {v}")
else:
    print("\nAll records were fully agreed — no debates were triggered.")

Records           : 50
Full agreement    : 46  (92.0%)  — debate skipped
Had disagreement  : 4  (8.0%)

Per-trait disagreement rate:
  Openness            : 2/50  (4.0%)
  Conscientiousness   : 1/50  (2.0%)
  Extraversion        : 2/50  (4.0%)
  Agreeableness       : 1/50  (2.0%)
  Neuroticism         : 0/50  (0.0%)

Example contested record  (idx=23)
Agreed on   : ['Openness', 'Conscientiousness', 'Agreeableness', 'Neuroticism']
Disagreed on: ['Extraversion']

Model A:
  Openness            : low
  Conscientiousness   : low
  Extraversion        : low ← disputed
  Agreeableness       : low
  Neuroticism         : high

Model B:
  Openness            : low
  Conscientiousness   : low
  Extraversion        : high ← disputed
  Agreeableness       : low
  Neuroticism         : high

Verifier final:
  Openness            : low
  Conscientiousness   : low
  Extraversion        : high
  Agreeableness       : low
  Neuroticism         : high


## Step 5 — Evaluate

In [12]:
_eval_model = f"{model_a_name}_vs_{model_b_name}_v_{verifier_model_name}"
_prompt_mode = "multi_llm_debate"

evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = _eval_model,
    run_time       = run_time,
    prompt_mode    = _prompt_mode,
    res_dir        = res_dir,
    run_id         = run_id,
)

print(f"\nFailed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(
    summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

Loaded predictions from F:\std\GR\code\model_x_ocean\result\debate\gpt-4o-mini_vs_gpt-4o-mini\20260518-211158\predictions.csv
Saved evaluation summary to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini_vs_gpt-4o-mini_v_gpt-4o-mini\multi_llm_debate\20260518-211158\evaluation_summary.csv
Saved Openness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini_vs_gpt-4o-mini_v_gpt-4o-mini\multi_llm_debate\20260518-211158\Openness_classification_report.txt
Saved Conscientiousness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini_vs_gpt-4o-mini_v_gpt-4o-mini\multi_llm_debate\20260518-211158\Conscientiousness_classification_report.txt
Saved Extraversion report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini_vs_gpt-4o-mini_v_gpt-4o-mini\multi_llm_debate\20260518-211158\Extraversion_classification_report.txt
Saved Agreeableness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini_vs_gpt-4o-mini_v_gpt-4o-mini\multi_llm_debate\20260518-211158\Agreeableness_classification_report.t

,trait,n_samples,accuracy,macro_f1,weighted_f1
0,Openness,50,0.68,0.500000,0.584000
1,Extraversion,50,0.64,0.506579,0.557895
2,Conscientiousness,50,0.50,0.365804,0.354135
3,Agreeableness,50,0.50,0.365804,0.354135
4,Neuroticism,50,0.48,0.324324,0.311351
